# Cleaning 2.3 - Clean energy calculations

This notebook does the following:
    (1) Creates variables so that the dataset can be merged with the equipment calculators
    (2) Merges the dataset with the equipment calculators
    (3) Performs the energy calculations

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from fill_missing_mode import fill_with_equipment_mode
from assign_set_temp import assign_set_temp

In [2]:
# Load data
equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_2.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# Load data dictionaries
equipment_data_dict = pd.read_excel(config.DATA_DICTIONARIES / "data_dictionary.xlsx", sheet_name="Equipment")

## (1) Prepare for merging with equipment calculators

In [4]:
# Clean number variable

# If number is missing, set number to 1
equipment.loc[equipment["number"].isna(), "number"] = 1

# Make number integer
equipment["number"] = equipment["number"].astype(int)


In [5]:
# Clean hours and days variables
# If hours is missing or "Unknown", set hours to 9
equipment.loc[(equipment["hours"].isna()) | (equipment["hours"] == "Unknown"), "hours"] = 9

# If days is missing or "Unknown", set days to 255
equipment.loc[(equipment["days"].isna()) | (equipment["days"] == "Unknown"), "days"] = 255

# Make hours and days numeric
equipment["hours"] = pd.to_numeric(equipment["hours"], errors="raise")
equipment["days"] = pd.to_numeric(equipment["days"], errors="raise")

In [6]:
# Clean hours_open variable

# Check unique values in hours_open
print("Unique values in hours_open before cleaning:")
print(equipment["hours_open"].unique())

# Fill missing/Unknown values for hours_open
equipment = fill_with_equipment_mode(
    equipment,
    value_col="hours_open",
    equipment_value="fc",
)

# Make hours_open numeric
equipment["hours_open"] = pd.to_numeric(equipment["hours_open"], errors="raise")

Unique values in hours_open before cleaning:
[nan  1.  2.  3.  8.  6.  4.  5.  0. 24.]


In [7]:
# Clean FC variables

# Create "Type" variable which is "VAV" if "controller_type" is "Variable Air Volume (VAV)" or "Unknown" or missing, and "CAV" if "controller_type" is "Constant Air Volume (CAV)" ONLY if equipment is "fc"
equipment["Type"] = np.where(
    (equipment["controller_type"].isin(["Variable Air Volume (VAV)", "Unknown"]) | equipment["controller_type"].isna()) 
     & (equipment["equipment"] == "fc"),
    "VAV",
    np.where((equipment["controller_type"] == "Constant Air Volume (CAV)") & (equipment["equipment"] == "fc"),
             "CAV",
             None)
)

In [14]:
# Clean water bath variables

# Fill missing/Unknown values for capacity_bath, heating, temp_bath, lid
for col in ["capacity_bath", "heating", "temp_bath", "lid"]:
    equipment = fill_with_equipment_mode(
        equipment,
        value_col=col,
        equipment_value="bath",
    )

# Create "Size" variable which is the same as "capacity_bath" except combining "10-12L" and "6-10L" into "10L"
equipment["Size"] = equipment["capacity_bath"].replace({"10-12L": "10L", "6-10L": "10L"})

# Set Set_Temp only for water bath rows
bath_mask = equipment["equipment"] == "bath"
equipment.loc[bath_mask, "Set_Temp"] = equipment.loc[bath_mask, "temp_bath"].apply(
    lambda t: assign_set_temp(t, [37, 65, 90])
)

In [ ]:
# Clean cryostat variables

# Fill missing/Unknown values for temp_cryostat and sleep_mode
for col in ["temp_cryostat", "sleep_mode"]:
    equipment = fill_with_equipment_mode(
        equipment,
        value_col=col,
        equipment_value="cryostat",
    )

# Print unique values in sleep_mode
print(equipment["sleep_mode"].unique())

# Fill in "Type" variable with "Energy Saving Mode Available"/"No Energy Saving Mode Available" based on sleep_mode
equipment["Type"] = np.where(
    (equipment["sleep_mode"].isin([
        "Yes, we use the unit 8am to 5pm, outside these hours it's in sleep mode",
        "Yes, but it's not used",
        "Yes, we use the unit 9am to 5pm, outside these hours it's in sleep mode"
        ]) | equipment["sleep_mode"].isna())
        & (equipment["equipment"] == "cryostat"),
    "Energy Saving Mode Available",
    np.where((equipment["sleep_mode"] == "No, it does not")
             & (equipment["equipment"] == "cryostat"),
             "No Energy Saving Mode Available",
             equipment["Type"])
)

# Set Set_Temp only for cryostat rows
cryostat_mask = equipment["equipment"] == "cryostat"
equipment.loc[cryostat_mask, "Set_Temp"] = equipment.loc[cryostat_mask, "temp_cryostat"].apply(
    lambda t: assign_set_temp(t, [-25, -20, -18])
)

[nan 'No, it does not'
 "Yes, we use the unit 8am to 5pm, outside these hours it's in sleep mode"
 "Yes, but it's not used"
 "Yes, we use the unit 9am to 5pm, outside these hours it's in sleep mode"]


In [ ]:
# Clean it variables

# Check unique values of it_type
print(equipment["it_type"].unique())

# Fill missing/Unknown values for it_type
equipment = fill_with_equipment_mode(
    equipment,
    value_col="it_type",
    equipment_value="it",
)


[nan 'High-powered desktop computer with graphics card' 'Standard laptop'
 'Standard desktop computer' 'High-powered laptop with graphics card']


## Clean up and save processed data

In [ ]:
# Save processed data
equipment.to_csv(config.PROCESSED_DATA / "panel_processed_3.csv", index =False)